In [1]:
# Cell 1: 环境准备
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from IPython.display import display, Markdown, HTML

from config import get_driver
driver = get_driver()

def h1(t): display(Markdown(f"<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>{t}</h1>"))
def h2(t): display(Markdown(f"<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ {t}</h2>"))
def h3(t): display(Markdown(f"<h3 style='color:#334155;margin-top:1rem;'>• {t}</h3>"))
def hr():    display(Markdown("---"))
def note(t): display(Markdown(f"<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>{t}</blockquote>"))
def table(rows, cols):
    if not rows:
        display(Markdown("_(无数据)_"))
        return
    lines = ["| " + " | ".join(str(c) for c in cols) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for r in rows:
        lines.append("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |")
    display(Markdown("\n".join(lines)))

print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# Cell 2: 演示一 —— 竞争对手情报
from src.ci.competitor_profile import build_competitor_profile, list_organizations
from src.ci.competitor_brief import generate_competitor_brief

h1("演示一：竞争对手情报")
note("选择一家竞争对手 → 展示公司档案、全部公开管线、目标病原、适应症、近期事件、原始来源 → 生成 Competitor Brief → 展示机会 / 威胁 / 不确定性和建议下一步")

# 1. 选一个覆盖最丰富的组织（有项目 + 有事件 + 有专利族）
with driver.session() as s:
    target = s.run("""
        MATCH (o:Organization)-[:DEVELOPS]->(:DevelopmentProgram)
        OPTIONAL MATCH (o)-[:OWNS_PATENT]->(pf:PatentFamily)
        OPTIONAL MATCH (o)<-[:CONCERNS]-(e:IntelligenceEvent)
        WITH o, count(DISTINCT pf) AS pc, count(DISTINCT e) AS ec
        WHERE ec > 0
        RETURN o.organization_id AS id, o.canonical_name AS name, pc, ec
        ORDER BY (pc + ec) DESC LIMIT 1
    """).single()

if not target:
    # 回退：只要有项目的组织
    orgs = list_organizations(driver)
    target = {"id": orgs[0]["id"], "name": orgs[0]["name"]}

org_id = target["id"]
org_name = target["name"]

h2(f"选中竞争对手：{org_name}")
note(f"organization_id = {org_id}")

# 2. 生成完整档案
profile = build_competitor_profile(driver, org_id)
if "error" in profile:
    raise RuntimeError(profile["error"])

# --- 公司基本信息 ---
h3("① 公司档案")
info = profile["organization"]
table(
    [{"字段": k, "值": v} for k, v in [
        ("名称", info.get("name")),
        ("类型", info.get("org_type")),
        ("国家", info.get("country")),
        ("网站", info.get("website")),
        ("状态", info.get("status")),
        ("上市/私有", info.get("public_private")),
    ]],
    ["字段", "值"],
)

# --- 管线 ---
h3("② 公开研发管线")
prog_rows = []
for p in profile.get("active_programs", []):
    pathogens = ", ".join(x["species"] for x in p.get("target_pathogens", []) if x.get("species"))
    prog_rows.append({
        "项目": p.get("name"),
        "阶段": p.get("stage"),
        "状态": p.get("status"),
        "模态": p.get("modality"),
        "靶向病原": pathogens or "—",
    })
table(prog_rows, ["项目", "阶段", "状态", "模态", "靶向病原"])

# --- 专利族 ---
h3("③ 专利布局")
patents = profile.get("patent_families", [])
table(
    [{
        "专利族ID": pf.get("patent_family_id"),
        "标题": (pf.get("title") or "")[:50],
        "法律状态": pf.get("legal_status"),
        "优先权日": pf.get("priority_date"),
        "法域": ", ".join(pf.get("jurisdictions") or []),
    } for pf in patents],
    ["专利族ID", "标题", "法律状态", "优先权日", "法域"],
)
if not patents:
    note("尚未收集到专利族")

# --- 临床试验 ---
h3("④ 临床试验")
trials = profile.get("clinical_trials", [])
table(
    [{
        "NCT号": t.get("nct_id"),
        "阶段": t.get("phase"),
        "状态": t.get("status"),
        "适应症": t.get("indication"),
        "标题": (t.get("title") or "")[:40],
    } for t in trials],
    ["NCT号", "阶段", "状态", "适应症", "标题"],
)
if not trials:
    note("尚未收集到临床试验")

# --- 近期事件 + 原始来源 ---
h3("⑤ 近期情报事件（含原始来源）")
events = profile.get("recent_events", [])[:8]
with driver.session() as s:
    event_rows = []
    for e in events:
        src = s.run("""
            MATCH (e:IntelligenceEvent {event_id: $eid})-[:HAS_SOURCE]->(src:SourceArtifact)
            RETURN src.source_id AS sid, src.title AS title, src.url AS url
            LIMIT 1
        """, eid=e["event_id"]).single()
        event_rows.append({
            "日期": e.get("event_date"),
            "类型": e.get("event_type"),
            "标题": (e.get("title") or "")[:46],
            "来源": src["sid"] if src else "—",
            "URL": (src["url"][:30] + "...") if src and src["url"] and len(src["url"]) > 30 else (src["url"] if src else "—"),
        })
table(event_rows, ["日期", "类型", "标题", "来源", "URL"])

# 3. 生成 Competitor Brief
h2("⑥ 生成 Competitor Brief")
brief = generate_competitor_brief(driver, org_id, days_back=730, persist=True)
bid = brief.get("brief_id")
note(f"brief_id = **{bid}**")

# --- 竞争信号 ---
ca = brief.get("competitive_assessment", {})
h3("⑦ 竞争评估")

def _render_signals(title, items):
    display(Markdown(f"**{title}**"))
    if not items:
        display(Markdown("_(无)_"))
        return
    for it in items:
        display(Markdown(
            f"- `{it.get('signal_date') or '—'}` · {it.get('signal_text')} "
            f"_(impact={it.get('impact_level')}, confidence={it.get('confidence')})_"
        ))

_render_signals("🚨 威胁（threats）", ca.get("threats", []))
_render_signals("💡 机会（opportunities）", ca.get("opportunities", []))
_render_signals("❓ 不确定性（uncertainties）", ca.get("uncertainties", []))

# --- 建议下一步 ---
h3("⑧ 建议下一步")
for s in brief.get("recommended_next_steps", []):
    display(Markdown(f"- {s}"))

# --- 数据缺口 ---
h3("⑨ 数据缺口")
for g in brief.get("data_gaps", []):
    display(Markdown(f"- ⚠️ {g}"))

hr()

<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>演示一：竞争对手情报</h1>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>选择一家竞争对手 → 展示公司档案、全部公开管线、目标病原、适应症、近期事件、原始来源 → 生成 Competitor Brief → 展示机会 / 威胁 / 不确定性和建议下一步</blockquote>

<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ 选中竞争对手：Adaptive Phage Therapeutics</h2>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>organization_id = CI:ORG:55DF2E8C</blockquote>

<h3 style='color:#334155;margin-top:1rem;'>• ① 公司档案</h3>

...

- 对手有 1 项活跃临床试验，建议持续跟踪

- 补充数据缺口，完善竞争对手档案

<h3 style='color:#334155;margin-top:1rem;'>• ⑨ 数据缺口</h3>

- ⚠️ 公司类型（上市/私有）未确认

---

In [3]:
# Cell 3: 演示二 —— 工程化噬菌体专业情报
from src.engineering_intelligence.strategy_classifier import get_all_strategies
from src.engineering_intelligence.technology_brief import generate_technology_brief

h1("演示二：工程化噬菌体专业情报")
note("选择「宿主范围扩展」策略 → 展示相关 EngineeringStrategy → 公开构建体、公司、项目、专利、临床试验 → 区分 TechnicalClaim 与 TechnicalResult → 比较证据成熟度 → 生成 Technology Brief → 展示技术空白、能力差距和待验证问题")

# 1. 选一个策略（优先选有构建体的）
strategies = get_all_strategies(driver)
if not strategies:
    raise RuntimeError("尚无工程策略，请先跑 06 notebook")

# 优先挑有专利族关联的策略
with driver.session() as s:
    pri = s.run("""
        MATCH (es:EngineeringStrategy)<-[:IMPLEMENTS]-(ec:EngineeredPhageConstruct)
        RETURN es.strategy_id AS sid, count(DISTINCT ec) AS cnt
        ORDER BY cnt DESC LIMIT 1
    """).single()
    if pri:
        target_sid = pri["sid"]
        target_stype = next((x["strategy_type"] for x in strategies if x["id"] == target_sid), "unknown")
    else:
        target_sid = strategies[0]["id"]
        target_stype = strategies[0]["strategy_type"]

h2(f"选中策略：{target_stype}")
note(f"strategy_id = {target_sid}")

# 2. 生成 Technology Brief
brief = generate_technology_brief(
    driver,
    strategy_id=target_sid,
    technology_question=f"{target_stype} 的公开证据成熟度如何？",
    persist=True,
)
bid = brief.get("brief_id")
note(f"brief_id = **{bid}**")

# --- 相关构建体 ---
h3("① 公开工程构建体")
constructs = brief.get("known_constructs", [])
table(
    [{
        "构建体": c.get("public_name"),
        "状态": c.get("construct_status"),
        "策略": ", ".join(c.get("strategy_types") or []),
        "靶向病原": ", ".join(c.get("target_pathogens") or []),
        "首次公开": c.get("first_public_date"),
    } for c in constructs],
    ["构建体", "状态", "策略", "靶向病原", "首次公开"],
)

# --- 目标病原 ---
h3("② 目标病原")
table(
    [{
        "病原": p.get("species"),
        "耐药机制": p.get("resistance_mechanism") or "—",
        "关联构建体数": p.get("construct_count"),
    } for p in brief.get("target_pathogens", [])],
    ["病原", "耐药机制", "关联构建体数"],
)

# --- 技术主张（Claim） ---
h3("③ 技术主张（TechnicalClaim）—— 来源声称什么")
table(
    [{
        "Claim ID": c.get("claim_id"),
        "类型": c.get("claim_type"),
        "主张": (c.get("claim_text") or "")[:52],
        "来源方": c.get("claimant_type"),
        "上下文": c.get("evidence_context"),
        "状态": c.get("claim_status"),
    } for c in brief.get("technical_claims", [])],
    ["Claim ID", "类型", "主张", "来源方", "上下文", "状态"],
)

# --- 技术结果（Result） ---
h3("④ 技术结果（TechnicalResult）—— 来源实际报告了什么")
table(
    [{
        "Result ID": r.get("result_id"),
        "类型": r.get("result_type"),
        "研究阶段": r.get("study_context"),
        "方向": r.get("outcome_direction"),
        "指标": f"{r.get('metric_name')}={r.get('metric_value')}{r.get('metric_unit') or ''}",
        "样本量": r.get("sample_size"),
        "可重现性": r.get("reproducibility_status"),
    } for r in brief.get("technical_results", [])],
    ["Result ID", "类型", "研究阶段", "方向", "指标", "样本量", "可重现性"],
)

note("★ Claim ≠ Result：主张是「来源声称什么」，结果是「来源实际报告了什么数据」，二者严格分开存储。")

# --- 专利族 ---
h3("⑤ 相关专利族")
table(
    [{
        "专利族ID": pf.get("patent_family_id"),
        "标题": (pf.get("title") or "")[:45],
        "法律状态": pf.get("legal_status"),
        "优先权日": pf.get("priority_date"),
        "权利人": ", ".join(pf.get("owners") or []),
    } for pf in brief.get("patent_families", [])],
    ["专利族ID", "标题", "法律状态", "优先权日", "权利人"],
)

# --- 证据成熟度 ---
h3("⑥ 证据成熟度分布")
em = brief.get("evidence_maturity", {})
dist = em.get("distribution", {})
display(Markdown(
    f"- **最高成熟度**：`{em.get('highest_maturity')}`  \n"
    f"- **分布**：`{dist}`  \n"
    f"- 说明：conceptual（概念）< in_vitro（体外）< in_vivo（体内）< clinical（临床）"
))

# --- 内部能力差距 ---
h3("⑦ 内部能力差距")
gaps = brief.get("internal_capability_gaps", [])
if gaps:
    table(
        [{
            "对象": f"{g['subject_type']}: {g['subject_id']}",
            "差距等级": g.get("gap_level"),
            "摘要": (g.get("summary") or "")[:60],
        } for g in gaps],
        ["对象", "差距等级", "摘要"],
    )
else:
    note("未发现明确的能力差距")

# --- 安全 / 可制造性 ---
h3("⑧ 安全与可制造性风险")
for label, key in [("安全性", "safety_uncertainties"), ("可制造性", "manufacturing_uncertainties")]:
    items = brief.get(key, [])
    if not items:
        display(Markdown(f"**{label}**：_(无)_"))
        continue
    display(Markdown(f"**{label}**"))
    for it in items:
        display(Markdown(f"- `{it.get('level')}` · {it.get('subject_type')} {it.get('subject_id')}"))

# --- 建议下一步 ---
h3("⑨ 建议下一步")
for s in brief.get("recommended_next_steps", []):
    display(Markdown(f"- {s}"))

hr()

<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>演示二：工程化噬菌体专业情报</h1>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>选择「宿主范围扩展」策略 → 展示相关 EngineeringStrategy → 公开构建体、公司、项目、专利、临床试验 → 区分 TechnicalClaim 与 TechnicalResult → 比较证据成熟度 → 生成 Technology Brief → 展示技术空白、能力差距和待验证问题</blockquote>

<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ 选中策略：host_range_expansion</h2>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>strategy_id = ENG:STRAT:33F089EB</blockquote>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>brief_id = **CI:BRIEF:7AF50CD1**</blockquote>

...

**可制造性**

- `medium` · strategy ENG:STRAT:33F089EB

<h3 style='color:#334155;margin-top:1rem;'>• ⑨ 建议下一步</h3>

- 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

---

In [4]:
# Cell 4: 演示三 —— 情报到决策
from src.decision_support.intelligence_question import (
    create_intelligence_question, list_intelligence_questions, link_brief_to_question,
)
from src.decision_support.watchlist_service import (
    create_watchlist_item, list_watchlist_items,
)
from src.shared.review import create_review
from src.decision_support.decision_record import create_decision_record
from src.ci.use_event_service import record_intelligence_use

h1("演示三：情报到决策")
note("IntelligenceQuestion → 系统组织（竞争对手布局 + 公开构建体 + 论文结果 + 专利密度 + 证据成熟度 + 内部能力差距）→ Decision Brief → 专家审核 → Decision Record")

# 1. 提出 IntelligenceQuestion
h2("① 提出情报问题")
q_text = "是否应优先评估针对目标病原菌的 biofilm_disruption 工程策略？"
iq_id = create_intelligence_question(
    driver,
    question_text=q_text,
    question_type="technology",
    requester_id="VP_Strategy",
    priority="high",
    actor_id="demo",
)
display(Markdown(f"- **问题**：{q_text}"))
display(Markdown(f"- **question_id**：`{iq_id}`"))

# 2. 系统组织信息
h2("② 系统组织的信息")

# 找一份此前的 Technology Brief（若有）
with driver.session() as s:
    tech_brief = s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'engineered_phage_technology'})
        RETURN b.brief_id AS bid, b.title AS title
        ORDER BY b.created_at DESC LIMIT 1
    """).single()

    comp_brief = s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'competitor'})
        RETURN b.brief_id AS bid, b.title AS title
        ORDER BY b.created_at DESC LIMIT 1
    """).single()

    # 专利密度
    patent_density = s.run("""
        MATCH (pf:PatentFamily)
        RETURN count(pf) AS cnt
    """).single()["cnt"]

    # 临床试验数
    trial_cnt = s.run("MATCH (ct:ClinicalTrial) RETURN count(ct) AS cnt").single()["cnt"]

    # 构建体数（biofilm_disruption）
    ec_cnt = s.run("""
        MATCH (es:EngineeringStrategy {strategy_type: 'biofilm_disruption'})
              <-[:IMPLEMENTS]-(ec)
        RETURN count(ec) AS cnt
    """).single()["cnt"]

h3("2.1 竞争对手布局")
if comp_brief:
    display(Markdown(f"- 竞争情报简报：`{comp_brief['bid']}`（{comp_brief['title']}）"))
else:
    display(Markdown("- _尚无竞争对手简报_"))

h3("2.2 公开构建体")
display(Markdown(f"- `biofilm_disruption` 策略下公开构建体数：**{ec_cnt}**"))

h3("2.3 专利密度")
display(Markdown(f"- 数据库中当前专利族总数：**{patent_density}**"))

h3("2.4 临床试验活跃度")
display(Markdown(f"- 数据库中当前临床试验总数：**{trial_cnt}**"))

h3("2.5 技术情报简报")
if tech_brief:
    display(Markdown(f"- 技术简报：`{tech_brief['bid']}`（{tech_brief['title']}）"))
else:
    display(Markdown("- _尚无技术简报_"))

# 3. 生成 Decision Brief（这里复用 IntelligenceProduct 作为简报载体）
h2("③ 生成 Decision Brief")
from src.ci.intelligence_product_service import create_intelligence_product

exec_summary = (
    f"本简报围绕问题「{q_text}」组织信息。"
    f"当前数据库中 {ec_cnt} 个公开构建体覆盖 biofilm_disruption 策略，"
    f"专利族 {patent_density} 族，临床试验 {trial_cnt} 项。"
    f"建议由情报分析师与领域专家会审，进入 FTO 初筛与可行性评估流程。"
)

decision_brief_id = create_intelligence_product(
    driver,
    brief_type="technology",
    title=f"决策简报 - {q_text[:40]}",
    executive_summary=exec_summary,
    as_of_date="2026-09-27",
    citations=[],
    competitive_assessment={},
    data_gaps=["部分策略尚缺乏体内数据"],
    recommended_next_steps=[
        "启动 FTO 初筛",
        "评估内部 biofilm 工程能力差距",
        "安排领域专家会审",
    ],
    actor_id="demo",
)
display(Markdown(f"- **brief_id**：`{decision_brief_id}`"))
display(Markdown(f"- **执行摘要**：{exec_summary}"))

# 把 brief 关联到 question
link_brief_to_question(driver, decision_brief_id, iq_id, actor_id="demo")
display(Markdown(f"- ✅ 简报 `{decision_brief_id}` 已回答问题 `{iq_id}`"))

# 4. 专家审核
h2("④ 专家审核")
review_id = create_review(
    driver,
    review_type="intelligence_product_review",
    target_object_type="IntelligenceProduct",
    target_object_id=decision_brief_id,
    reviewer_id="expert_wang",
    decision="approved",
    comment="技术论证充分，同意进入 FTO 初筛",
    update_target_status=True,
    actor_id="demo",
)
display(Markdown(f"- **Review ID**：`{review_id}`"))
display(Markdown(f"- **审核决策**：approved"))
display(Markdown(f"- **审核人**：expert_wang"))

# 5. DecisionRecord
h2("⑤ 决策记录")
dec_id = create_decision_record(
    driver,
    brief_id=decision_brief_id,
    decision_type="evaluate",
    decision_summary="启动 biofilm_disruption 工程策略的可行性评估，并同步启动 FTO 初筛",
    rationale=f"基于 {decision_brief_id} 与专家审核 {review_id}",
    decision_owner="VP_Strategy",
    review_date="2027-03-01",
    actor_id="demo",
)
display(Markdown(f"- **decision_id**：`{dec_id}`"))
display(Markdown(f"- **决策类型**：evaluate"))
display(Markdown(f"- **决策摘要**：启动 biofilm_disruption 工程策略的可行性评估，并同步启动 FTO 初筛"))
display(Markdown(f"- **负责人**：VP_Strategy"))
display(Markdown(f"- **复核日期**：2027-03-01"))

# 6. 记录消费
h2("⑥ 记录情报消费")
use_id = record_intelligence_use(
    driver,
    product_id=decision_brief_id,
    consumer_type="Strategy",
    consumer_id="strategy_team",
    use_purpose="go_no_go_decision",
    context_note="战略团队基于简报做出可行性评估决策",
    referenced_decision_id=dec_id,
    actor_id="demo",
)
display(Markdown(f"- **use_event_id**：`{use_id}`"))

# 7. 加入 Watchlist
h2("⑦ 加入关注名单")
with driver.session() as s:
    strategy = s.run("""
        MATCH (es:EngineeringStrategy {strategy_type: 'biofilm_disruption'})
        RETURN es.strategy_id AS sid LIMIT 1
    """).single()

if strategy:
    wi_id = create_watchlist_item(
        driver,
        target_type="engineering_strategy",
        target_id=strategy["sid"],
        reason="biofilm_disruption 策略公开证据增长，需持续跟踪",
        priority="high",
        owner_id="R&D_Lead",
        review_date="2027-03-01",
        derived_from_brief_id=decision_brief_id,
        actor_id="demo",
    )
    display(Markdown(f"- **watchlist_item_id**：`{wi_id}`"))
    display(Markdown(f"- **理由**：biofilm_disruption 策略公开证据增长，需持续跟踪"))

# 8. 归因链验证
h2("⑧ 归因链验证")
with driver.session() as s:
    chain = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        OPTIONAL MATCH (b)-[:ANSWERS]->(iq:IntelligenceQuestion)
        OPTIONAL MATCH (b)<-[:REVIEWS]-(r:Review)
        OPTIONAL MATCH (dr:DecisionRecord)-[:BASED_ON]->(b)
        OPTIONAL MATCH (u:IntelligenceUseEvent)-[:CONSUMES]->(b)
        OPTIONAL MATCH (wi:WatchlistItem)-[:DERIVED_FROM]->(b)
        RETURN
            iq.question_text AS question,
            b.brief_id AS brief,
            r.decision AS review_decision,
            dr.decision_id AS decision,
            dr.decision_type AS decision_type,
            u.consumer_type AS consumer,
            u.use_purpose AS purpose,
            wi.watchlist_item_id AS watchlist
    """, bid=decision_brief_id).single()

if chain:
    table(
        [{
            "问题": (chain["question"] or "")[:40],
            "简报": chain["brief"],
            "审核": chain["review_decision"],
            "决策": f"{chain['decision']} ({chain['decision_type']})",
            "消费方": chain["consumer"],
            "用途": chain["purpose"],
            "关注项": chain["watchlist"],
        }],
        ["问题", "简报", "审核", "决策", "消费方", "用途", "关注项"],
    )

hr()

# 9. 总结
h1("演示总结")
display(Markdown("""
本次演示展示了从**外部公开情报**到**组织内部决策**的完整闭环：
外部公开信号
↓
SourceArtifact（可追溯来源）
↓
IntelligenceEvent / TechnicalClaim / TechnicalResult（事实层）
↓
TechnologyAssessment / CompetitorAssessment（分析判断层）
↓
DecisionBrief + IntelligenceQuestion（决策辅助）
↓
专家审核（Review）
↓
DecisionRecord + IntelligenceUseEvent + WatchlistItem（决策层）
↓
后续结果跟踪
**关键设计原则**：
- 所有情报可追溯（每个对象都能回溯到 SourceArtifact）
- 事实与判断分离（Fact Layer / Assessment Layer 物理隔离）
- 所有结论有时间属性（每个 Brief 都有 as_of_date）
- 所有决策可回溯（DecisionRecord → Brief → 事实 → 来源）
"""))

<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>演示三：情报到决策</h1>

<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>IntelligenceQuestion → 系统组织（竞争对手布局 + 公开构建体 + 论文结果 + 专利密度 + 证据成熟度 + 内部能力差距）→ Decision Brief → 专家审核 → Decision Record</blockquote>

<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ ① 提出情报问题</h2>

- **问题**：是否应优先评估针对目标病原菌的 biofilm_disruption 工程策略？

- **question_id**：`CI:IQ:7B26F4BC`

...

<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ ⑧ 归因链验证</h2>

| 问题 | 简报 | 审核 | 决策 | 消费方 | 用途 | 关注项 |
|---|---|---|---|---|---|---|
| 是否应优先评估针对目标病原菌的 biofilm_disruption 工程策略？ | CI:BRIEF:E32179AF | approved | CI:DEC:5C4227C0 (evaluate) | Strategy | go_no_go_decision | CI:WATCH:7ABD7224 |

---

<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>演示总结</h1>


本次演示展示了从**外部公开情报**到**组织内部决策**的完整闭环：
外部公开信号
↓
SourceArtifact（可追溯来源）
↓
IntelligenceEvent / TechnicalClaim / TechnicalResult（事实层）
↓
TechnologyAssessment / CompetitorAssessment（分析判断层）
↓
DecisionBrief + IntelligenceQuestion（决策辅助）
↓
专家审核（Review）
↓
DecisionRecord + IntelligenceUseEvent + WatchlistItem（决策层）
↓
后续结果跟踪
**关键设计原则**：
- 所有情报可追溯（每个对象都能回溯到 SourceArtifact）
- 事实与判断分离（Fact Layer / Assessment Layer 物理隔离）
- 所有结论有时间属性（每个 Brief 都有 as_of_date）
- 所有决策可回溯（DecisionRecord → Brief → 事实 → 来源）
